# 02 — Model Eğitimi
Hibrit QNN modelini adım adım eğitir ve eğitim sürecini görselleştirir.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
from src.quantum_model import HybridQNN, load_config
from src.train import get_dataloaders, train_epoch, eval_epoch

config = load_config()
device = torch.device('cpu')
model  = HybridQNN.from_config(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Toplam parametre: {total_params:,}')

## Veri Yükle

In [ ]:
train_loader, test_loader = get_dataloaders(config)
print(f'Eğitim: {len(train_loader.dataset)} örnek')
print(f'Test  : {len(test_loader.dataset)} örnek')

# Örnek görüntüler
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(str(labels[i].item()))
    ax.axis('off')
plt.suptitle('MNIST — Eğitim Örnekleri')
plt.tight_layout()
plt.show()

## Eğitim (Kısa Demo — 3 Epoch)

In [ ]:
import torch.nn as nn

optimizer = torch.optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
criterion = nn.CrossEntropyLoss()

DEMO_EPOCHS = 3
log = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}

for epoch in range(1, DEMO_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    te_loss, te_acc = eval_epoch(model, test_loader, criterion, device)
    log['train_acc'].append(tr_acc)
    log['test_acc'].append(te_acc)
    log['train_loss'].append(tr_loss)
    log['test_loss'].append(te_loss)
    print(f'Epoch {epoch}/{DEMO_EPOCHS} | Train: %{tr_acc*100:.1f} | Test: %{te_acc*100:.1f}')

In [ ]:
epochs = range(1, DEMO_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, log['train_acc'], 'o-', label='Eğitim')
ax1.plot(epochs, log['test_acc'],  's-', label='Test')
ax1.set_title('Doğruluk')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, log['train_loss'], 'o-', label='Eğitim')
ax2.plot(epochs, log['test_loss'],  's-', label='Test')
ax2.set_title('Kayıp')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.suptitle('Eğitim Eğrileri')
plt.tight_layout()
plt.show()